# NB11 — synergy and the S4 ship/cut gate

Hold out **pairs**. Gate: Spearman vs ALMANAC ≥ 0.3 on held-out pairs where
**both** drugs are in-topology, **n ≥ 100**. Out-of-topology pairs are logged, not gated.


In [ ]:
from pathlib import Path
import sys, json, warnings
warnings.filterwarnings("ignore")

cwd = Path.cwd().resolve()
for cand in [cwd, *cwd.parents]:
    if (cand / "src" / "gate.py").is_file():
        sys.path.insert(0, str(cand / "src"))
        break
    nested = cand / "v2"
    if (nested / "src" / "gate.py").is_file():
        sys.path.insert(0, str(nested / "src"))
        break

from paths import ensure_src_on_path, resolve_v2_root
from gate import gate as _gate_impl
from safety import assert_safe
try:
    import certifi, os
    os.environ.setdefault("SSL_CERT_FILE", certifi.where())
    os.environ.setdefault("REQUESTS_CA_BUNDLE", certifi.where())
except Exception:
    pass

V2_ROOT = resolve_v2_root()
ensure_src_on_path(V2_ROOT)
REPO_ROOT = V2_ROOT.parent
RAW = V2_ROOT / "data" / "raw"
INTERIM = V2_ROOT / "data" / "interim"
REF = V2_ROOT / "data" / "reference"
ARTIFACTS = V2_ROOT / "artifacts"
FIGURES = V2_ROOT / "reports" / "figures"
for d in (RAW, INTERIM, REF, ARTIFACTS, FIGURES, INTERIM / "causal_networks"):
    d.mkdir(parents=True, exist_ok=True)

# Laptop vs VPS. Smoke passes are provisional until a full run converts them.
# NB01 and NB04 stay full: harmonisation and the VAE are cheap.
SMOKE_TEST = False
N_SAMPLES  = None   # full TCGA-BRCA; do not cap
N_SC_CELLS = 25_000  # Wu reference subsample if RAM is tight
N_PATIENTS = None
N_DRUGS    = None

def gate(*args, **kwargs):
    kwargs.setdefault("smoke_test", SMOKE_TEST)
    if "sample_ids" not in kwargs:
        kwargs.setdefault("cohort", False)
    return _gate_impl(*args, **kwargs)

print("V2_ROOT =", V2_ROOT, "SMOKE_TEST =", SMOKE_TEST)


In [ ]:
# Config
RHO_MIN = 0.3
MIN_N = 100
import json, numpy as np, pandas as pd
from scipy.stats import spearmanr
from sklearn.model_selection import train_test_split
from topology import default_topology
from ode_lib import make_rhs, drug_multiplier, simulate_euler, simulate_trajectory, detect_rebound, bliss_excess_from_effects
from pk_table import load_almanac_named_pairs, load_pk_table
from io_data import canon_drug
nodes = pd.read_csv(REF / "ode_nodes.csv")["gene"].tolist()
pk = load_pk_table(REF / "drug_pk.csv")
topo = json.loads((REF / "ode_topology.json").read_text()) if (REF / "ode_topology.json").exists() else default_topology(nodes)
params = np.load(ARTIFACTS / "ode_params.npz") if (ARTIFACTS / "ode_params.npz").exists() else None
s4 = json.loads((INTERIM / "NB10_s4_decision.json").read_text()) if (INTERIM / "NB10_s4_decision.json").exists() else {}
print("S4 decision from NB10", s4.get("decision"))


In [ ]:
# Load ALMANAC — raw ComboDrugGrowth or Q5 breast pair summary (no invented NSCs)
almanac = load_almanac_named_pairs(RAW / "almanac", REF)
print("ALMANAC named pairs", None if almanac is None else almanac.shape)


In [ ]:
# Compute Bliss excess for every pair in the PK table
idx = {g: i for i, g in enumerate(topo["nodes"])}
k = params["k"] if params is not None else np.full(len(topo["edges"]), 0.5)
tau = params["tau"] if params is not None else np.ones(len(topo["nodes"]))
rhs = make_rhs(topo, {"k": k, "tau": tau, "n": 2.0})
e2f = idx.get("E2F1", len(idx)-1)
x0 = np.full(len(idx), 0.5)
unt = simulate_euler(rhs, x0, np.ones(len(idx)), t_end=72)[e2f]

def effect(target, conc, ic50):
    m = drug_multiplier(idx[target], conc, ic50, n_nodes=len(idx))
    y = simulate_euler(rhs, x0, m, t_end=72)[e2f]
    return 1.0 - (y / (unt + 1e-8))

rows = pk[pk["target_gene"].isin(idx)].copy()
rows["cmax_nm"] = pd.to_numeric(rows["cmax_nm"], errors="coerce")
rows["ic50_nm"] = pd.to_numeric(rows["ic50_nm"], errors="coerce")
rows = rows.dropna(subset=["cmax_nm", "ic50_nm"]).reset_index(drop=True)
in_topo = set(pk.loc[pk["in_ode_topology"], "drug_name"].map(canon_drug))
pairs = []
for i in range(len(rows)):
    for j in range(i+1, len(rows)):
        a, b = rows.iloc[i], rows.iloc[j]
        if a["target_gene"] == b["target_gene"]:
            continue
        ea = effect(a["target_gene"], a["cmax_nm"], a["ic50_nm"])
        eb = effect(b["target_gene"], b["cmax_nm"], b["ic50_nm"])
        m = np.ones(len(idx))
        m[idx[a["target_gene"]]] = 1.0 / (1.0 + (a["cmax_nm"] / a["ic50_nm"]))
        m[idx[b["target_gene"]]] = 1.0 / (1.0 + (b["cmax_nm"] / b["ic50_nm"]))
        eab = 1.0 - simulate_euler(rhs, x0, m, t_end=72)[e2f] / (unt + 1e-8)
        pairs.append({
            "drug_a": a["drug"], "drug_b": b["drug"],
            "both_in_topology": canon_drug(a["drug"]) in in_topo and canon_drug(b["drug"]) in in_topo,
            "bliss_excess": bliss_excess_from_effects(ea, eb, eab),
        })
pair_df = pd.DataFrame(pairs)
if len(pair_df) >= 4:
    tr, te = train_test_split(pair_df.index, test_size=0.4, random_state=0)
else:
    te = pair_df.index
held = pair_df.loc[te]
# ALMANAC comparison when names overlap; else the gate fails honestly
rho = 0.0
n_join = 0
note = "no ALMANAC"
if almanac is not None and len(almanac) and len(held):
    obs = almanac.copy()
    obs["a"] = obs["drug_a"].map(canon_drug)
    obs["b"] = obs["drug_b"].map(canon_drug)
    keys = {}
    for _, r in obs.iterrows():
        keys[tuple(sorted((r["a"], r["b"])))] = float(r["score"])
    pred, truth, in_flags = [], [], []
    for _, r in held.iterrows():
        key = tuple(sorted((canon_drug(r["drug_a"]), canon_drug(r["drug_b"]))))
        if key in keys:
            pred.append(float(r["bliss_excess"]))
            truth.append(keys[key])
            in_flags.append(bool(r.get("both_in_topology", True)))
    n_join = len(pred)
    pred_in = [p for p, f in zip(pred, in_flags) if f]
    truth_in = [t for t, f in zip(truth, in_flags) if f]
    n_join_in = len(pred_in)
    if n_join_in >= 3:
        rho = float(spearmanr(pred_in, truth_in).statistic)
        if not np.isfinite(rho):
            rho = 0.0
        note = f"held-out BOTH-in-topology joined={n_join_in} all_joined={n_join} named_pairs={len(obs)}"
    elif n_join >= 3:
        rho = float(spearmanr(pred, truth).statistic)
        if not np.isfinite(rho):
            rho = 0.0
        note = f"in-topology join too thin n_join_in={n_join_in}; all-pairs rho logged n_join={n_join}"
    else:
        note = f"ALMANAC loaded but join too thin n_join={n_join} named_pairs={len(obs)}"
    n_join = n_join_in if n_join_in else n_join
pair_df.to_parquet(INTERIM / "predicted_synergy.parquet")
print(pair_df.head())
print("held-out pairs", len(held))

# Resistance / rebound on palbociclib -> CDK4
traj = simulate_trajectory(rhs, x0, drug_multiplier(idx.get("CDK4", 0), 200, 11, n_nodes=len(idx)), t_span=(0, 168))
rebound = detect_rebound(traj, topo["nodes"])
print("rebound nodes", rebound)
pd.Series(rebound).to_csv(INTERIM / "NB11_rebound_nodes.csv", index=False)


In [ ]:
# GATE — n≥100 on both-in-topology held-out pairs is the ship/cut decision
if almanac is None or almanac is not None and len(almanac) == 0:
    extra = "ALMANAC missing — cannot claim the ODE earns its complexity"
else:
    extra = note
extra = extra + f" | NB10_s4={s4.get('decision')}"
gate("NB11", "synergy_vs_almanac_heldout", float(rho), RHO_MIN,
     n=n_join if almanac is not None and len(almanac) else 0, min_n=MIN_N, smoke_test=False,
     note=f"n_pairs={len(pair_df)} n_join={n_join} {extra}")
print("If this fails or is insufficient, S4 is a paper section — no simulator panel.")


In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(5, 3))
if len(pair_df):
    ax.hist(pair_df["bliss_excess"], bins=20)
ax.set_title("Predicted Bliss excess")
fig.tight_layout(); fig.savefig(FIGURES / "NB11_bliss.png", dpi=140)
